## Homework 3: Recommender Systems
Owen Randolph, DSCI-590: Applied Data Science, Fall 2024

This notebook will walk through the creation of user-item and item-item collaborative-based filtering systems recommending the top 5 items to the user based on the choice of a previous user or an item, respectively.  A more sophisticated deep-learning model will be built using Neural Collaborative Filtering (NCF) to predict the rating of a given item by a given user.  I'll be using the electronics marketing bias dataset found at https://github.com/MengtingWan/marketBias/tree/master/data, cited at the end of this notebook. 

In [177]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity

In [178]:
#Load dataset and check
df = pd.read_csv("df_electronics.csv")
df.head()

,item_id,user_id,rating,timestamp,model_attr,category,brand,year,user_attr,split
0,0,0,5.0,1999-06-13,Female,Portable Audio & Video,NaN,1999,NaN,0
1,0,1,5.0,1999-06-14,Female,Portable Audio & Video,NaN,1999,NaN,0
2,0,2,3.0,1999-06-17,Female,Portable Audio & Video,NaN,1999,NaN,0
3,0,3,1.0,1999-07-01,Female,Portable Audio & Video,NaN,1999,NaN,0
4,0,4,2.0,1999-07-06,Female,Portable Audio & Video,NaN,1999,NaN,0


In [179]:
# Check the number of rows and columns of the dataset
df.shape

(1292954, 10)

In [180]:
# Check the data types in columns
df.dtypes

item_id         int64
user_id         int64
rating        float64
timestamp      object
model_attr     object
category       object
brand          object
year            int64
user_attr      object
split           int64
dtype: object

In [181]:
# Check Missing Values
df.isnull().sum()

item_id             0
user_id             0
rating              0
timestamp           0
model_attr          0
category            0
brand          961834
year                0
user_attr     1118830
split               0
dtype: int64

In [182]:
# Drop Feature variables with high number of missing values and categories that won't be used in recommendation
df1 = df.drop(['brand', 'split', 'model_attr'], axis=1)
df = df.drop(['brand', 'user_attr', 'split', 'timestamp', 'model_attr', 'year'], axis=1)
df.head()

,item_id,user_id,rating,category
0,0,0,5.0,Portable Audio & Video
1,0,1,5.0,Portable Audio & Video
2,0,2,3.0,Portable Audio & Video
3,0,3,1.0,Portable Audio & Video
4,0,4,2.0,Portable Audio & Video


In [183]:
# Filter out users that have given over 5 ratings
user_counts = df.user_id.value_counts()
users_more_than_one = user_counts[user_counts > 5].index
df = df[df.user_id.isin(users_more_than_one)]

In [184]:
df_sorted_desc = df.sort_values(by='user_id', ascending=False)
df_sorted_desc

,item_id,user_id,rating,category
1153371,6726,1034031,3.0,Camera & Photo
1153250,6600,1034031,4.0,Computers & Accessories
1153308,2586,1034031,3.0,Camera & Photo
1153269,2981,1034031,3.0,Camera & Photo
1153709,3404,1034031,4.0,Camera & Photo
...,...,...,...,...
14933,651,28,4.0,Camera & Photo
84919,1776,28,3.0,Camera & Photo
541805,7213,28,3.0,Headphones
79078,1857,28,5.0,Portable Audio & Video


In [185]:
# User Rating Frequency
df.user_id.value_counts()

user_id
142967     41
30661      38
89185      37
80476      34
46878      34
           ..
49146       6
239021      6
238968      6
237388      6
1034031     6
Name: count, Length: 1158, dtype: int64

In [186]:
# Show the number of users
user_unique = df.user_id.nunique()
print(user_unique)

1158


In [187]:
# Show the number of items
item_unique = df.item_id.nunique()
print(item_unique)

2883


In [188]:
# Product categories
print(df.category.unique())

['Portable Audio & Video' 'Camera & Photo' 'Computers & Accessories'
 'Headphones' 'Accessories & Supplies' 'Television & Video'
 'Car Electronics & GPS' 'Home Audio' 'Security & Surveillance'
 'Wearable Technology']


### User-Item Recommender System

In [189]:
# Create a matrix of ratings by user_id and item_id 
user_item_matrix = df.pivot_table(index='user_id', columns='item_id', values='rating').fillna(0)
# Create a user-item matrix
user_item_matrix = df.pivot_table(index='user_id', columns='item_id', values='rating').fillna(0)
user_item_matrix.head()

item_id,0,2,3,10,14,15,16,17,24,40,...,9429,9446,9468,9484,9503,9504,9509,9510,9516,9537
user_id,,,,,,,,,,,,,,,,,,,,,
28,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
158,0.0,0.0,2.0,0.0,4.0,0.0,3.0,0.0,5.0,4.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
269,0.0,0.0,0.0,0.0,5.0,5.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
789,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
822,0.0,0.0,0.0,5.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [190]:
# Calculate cosine similarity
user_similarity = cosine_similarity(user_item_matrix)

# Convert similarity matrix to DataFrame
user_similarity_df = pd.DataFrame(user_similarity, index=user_item_matrix.index, columns=user_item_matrix.index)
print("\nUser-User Similarity Matrix:")
user_similarity_df.head()


User-User Similarity Matrix:


user_id,28,158,269,789,822,1007,1156,1170,1188,1277,...,893097,899072,899955,907563,911696,919192,919932,926621,978504,1034031
user_id,,,,,,,,,,,,,,,,,,,,,
28,1.000000,0.000000,0.0000,0.039878,0.0,0.0,0.000000,0.0,0.0,0.000000,...,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0
158,0.000000,1.000000,0.0902,0.069296,0.0,0.0,0.091933,0.0,0.0,0.096943,...,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0
269,0.000000,0.090200,1.0000,0.000000,0.0,0.0,0.000000,0.0,0.0,0.000000,...,0.124546,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0
789,0.039878,0.069296,0.0000,1.000000,0.0,0.0,0.000000,0.0,0.0,0.000000,...,0.000000,0.0,0.0,0.0,0.0,0.183694,0.0,0.0,0.0,0.0
822,0.000000,0.000000,0.0000,0.000000,1.0,0.0,0.000000,0.0,0.0,0.000000,...,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0


In [191]:
def User_Item_Recommender(user_id, user_item_matrix, user_similarity_df, num_recommendations=5):
    # Get similarity scores for the input user
    similar_users = user_similarity_df[user_id].sort_values(ascending=False)

    # Get the user's ratings
    user_ratings = user_item_matrix.loc[user_id]

    # Identify items not rated by the user
    unrated_items = user_ratings[user_ratings == 0].index

    # Compute a weighted average of ratings of similar users for each unrated item
    scores = {}
    for item in unrated_items:
        total_score = 0
        total_similarity = 0
        for other_user in similar_users.index:
            if user_item_matrix.loc[other_user, item] > 0:  # Consider only users who rated the item
                total_score += user_similarity_df[user_id][other_user] * user_item_matrix.loc[other_user, item]
                total_similarity += user_similarity_df[user_id][other_user]

        # Avoid division by zero
        if total_similarity > 0:
            scores[item] = total_score / total_similarity

    # Sort and return the top recommendations
    recommended_items = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    return [item for item, score in recommended_items[:num_recommendations]]

In [192]:
# Example usage: Get recommendations for user 158
recommendations = User_Item_Recommender(user_id=158, user_item_matrix=user_item_matrix, user_similarity_df=user_similarity_df)
print(f"\nRecommended items for user 158: {recommendations}")


Recommended items for user 158: [81, 2569, 15, 43, 74]


In [193]:
# Possible users to get recommendations for
print("Users in the matrix:", user_similarity_df.index)

Users in the matrix: Index([     28,     158,     269,     789,     822,    1007,    1156,    1170,
          1188,    1277,
       ...
        893097,  899072,  899955,  907563,  911696,  919192,  919932,  926621,
        978504, 1034031],
      dtype='int64', name='user_id', length=1158)


### Item-Item Recommender System
This model will use a collaborative-based filtering algorithm to give a recommendation for an item based on other items that have been rated similarly to it.

In [194]:
item_user_matrix = df.pivot_table(index='item_id', columns='user_id', values='rating').fillna(0)

In [195]:
# Compute cosine similarity between items
item_similarity = cosine_similarity(item_user_matrix)
item_user_matrix.head()

user_id,28,158,269,789,822,1007,1156,1170,1188,1277,...,893097,899072,899955,907563,911696,919192,919932,926621,978504,1034031
item_id,,,,,,,,,,,,,,,,,,,,,
0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
10,0.0,0.0,0.0,0.0,5.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
14,0.0,4.0,5.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [196]:
# Convert the similarity matrix to a DataFrame
item_similarity_df = pd.DataFrame(item_similarity, index=item_user_matrix.index, columns=item_user_matrix.index)
print("\nItem-Item Similarity Matrix:")
item_similarity_df.head()


Item-Item Similarity Matrix:


item_id,0,2,3,10,14,15,16,17,24,40,...,9429,9446,9468,9484,9503,9504,9509,9510,9516,9537
item_id,,,,,,,,,,,,,,,,,,,,,
0,1.0,0.0,0.000000,0.0,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,...,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000
2,0.0,1.0,0.000000,0.0,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,...,0.0,0.0,0.101015,0.0,0.0,0.0,0.0,0.032397,0.0,0.000000
3,0.0,0.0,1.000000,0.0,0.279372,0.000000,0.447214,0.0,0.447214,0.357771,...,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.275241
10,0.0,0.0,0.000000,1.0,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,...,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000
14,0.0,0.0,0.279372,0.0,1.000000,0.780869,0.624695,0.0,0.624695,0.499756,...,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000


We create function called item_item_recommender() which recommends the top 5 items to the user based on their choice.
1. The code finds the numerical index of the chosen item in the matrix
2. It sorts the similarity scores for that item in descending order
3. It selects the top 5 most similar items
4. It retrieves the category of the recommended items
5. It returns the information as a list

In [197]:
def Item_Item_recommender(item_id, item_similarity_df, num_recommendations=5):
    # Check if the item_id exists in the similarity DataFrame
    if item_id not in item_similarity_df.index:
        print(f"Item {item_id} not found in the similarity matrix.")
        return []

    # Get similarity scores for the given item
    similar_items = item_similarity_df[item_id].sort_values(ascending=False)

    # Exclude the item itself from the recommendations
    similar_items = similar_items.drop(item_id)

    # Return the top N recommended items
    return similar_items.head(num_recommendations).index.tolist()

In [208]:
# Example of item-item recommender function in use for item 40
recommendations = Item_Item_recommender(item_id=40, item_similarity_df=item_similarity_df)
print(f"\nRecommended items for item 40: {recommendations}")


Recommended items for item 40: [16, 24, 63, 46, 730]


### Recommendation based on Deep Learning
This will be a different approach using deep learning techniques using Neural Collaborative Filtering (NCF).  A version of the dataset that has more features will be used in order to capture more diversity on user-item interaction and preferences.

In [199]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Embedding, Flatten, Concatenate, Dense

In [200]:
df.head()

,item_id,user_id,rating,category
28,0,28,2.0,Portable Audio & Video
158,3,158,2.0,Camera & Photo
183,14,158,4.0,Camera & Photo
271,14,269,5.0,Camera & Photo
279,15,269,5.0,Camera & Photo


In [201]:
df.shape

(9509, 4)

In [202]:
# Encode user_id and item_id as integers
user_encoder = LabelEncoder()
item_encoder = LabelEncoder()
df['user_id'] = user_encoder.fit_transform(df['user_id'])
df['item_id'] = item_encoder.fit_transform(df['item_id'])

In [203]:
# Train Test Split
train, test = train_test_split(df, test_size=0.2, random_state=42)

In [209]:
from tensorflow.keras.layers import Dense
# Input layers
user_input = Input(shape=(1,), name='user_input')
item_input = Input(shape=(1,), name='item_input')

# Output layer with a sigmoid activation scaled to range 0-5
output = Dense(1, activation='sigmoid')(dense) 
scaled_output = output * 5  

# Embedding layers
user_embedding = Embedding(input_dim=user_unique, output_dim=50, name='user_embedding')(user_input)
item_embedding = Embedding(input_dim=item_unique, output_dim=50, name='item_embedding')(item_input)

# Flatten embeddings
user_vec = Flatten()(user_embedding)
item_vec = Flatten()(item_embedding)

# Concatenate user and item vectors
concat = Concatenate()([user_vec, item_vec])

# Dense layers
dense = Dense(128, activation='relu')(concat)
dense = Dense(64, activation='relu')(dense)
output = Dense(1, activation='linear')(dense)

# Build and compile model
model = Model(inputs=[user_input, item_input], outputs=output)
model.compile(optimizer='adam', loss='mean_squared_error')

#### Train the Model

In [210]:
history = model.fit(
    [train['user_id'], train['item_id']],  # Input user and item IDs
    train['rating'],                       # Target ratings
    epochs=10,                             # Number of epochs (adjust as needed)
    batch_size=32,                         # Batch size (adjust as needed)
    validation_split=0.1,                  # Use 10% of training data for validation
    verbose=1
)

Epoch 1/10
214/214 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 9.8148 - val_loss: 0.9838
Epoch 2/10
214/214 ━━━━━━━━━━━━━━━━━━━━ 0s 868us/step - loss: 0.8086 - val_loss: 0.9961
Epoch 3/10
214/214 ━━━━━━━━━━━━━━━━━━━━ 0s 786us/step - loss: 0.5631 - val_loss: 1.0667
Epoch 4/10
214/214 ━━━━━━━━━━━━━━━━━━━━ 0s 748us/step - loss: 0.4883 - val_loss: 1.1051
Epoch 5/10
214/214 ━━━━━━━━━━━━━━━━━━━━ 0s 789us/step - loss: 0.4195 - val_loss: 1.1539
Epoch 6/10
214/214 ━━━━━━━━━━━━━━━━━━━━ 0s 847us/step - loss: 0.3420 - val_loss: 1.1479
Epoch 7/10
214/214 ━━━━━━━━━━━━━━━━━━━━ 0s 750us/step - loss: 0.2943 - val_loss: 1.2103
Epoch 8/10
214/214 ━━━━━━━━━━━━━━━━━━━━ 0s 750us/step - loss: 0.2423 - val_loss: 1.2877
Epoch 9/10
214/214 ━━━━━━━━━━━━━━━━━━━━ 0s 776us/step - loss: 0.1759 - val_loss: 1.3600
Epoch 10/10
214/214 ━━━━━━━━━━━━━━━━━━━━ 0s 761us/step - loss: 0.1261 - val_loss: 1.3568


In [211]:
# Evaluate the model on the test data
test_loss = model.evaluate([test['user_id'], test['item_id']], test['rating'])
print(f"Test Loss: {test_loss}")

60/60 ━━━━━━━━━━━━━━━━━━━━ 0s 319us/step - loss: 1.2515
Test Loss: 1.2955105304718018


In [212]:
# Example of a Prediction
# Convert inputs to numpy arrays before transforming
user_id = np.array([822])  # Convert to NumPy array
item_id = np.array([98])   # Convert to NumPy array

# Perform prediction
predicted_rating = model.predict([user_encoder.transform(user_id), item_encoder.transform(item_id)])
print(f"Predicted Rating: {predicted_rating}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 60ms/step
Predicted Rating: [[4.673868]]


This model is not hyper-parameter tuned.  Although training loss decreased continuously throughout the training, validation loss sees fluctuation, indicating possible overfitting of the model.  We could introduce L2 regularization or drop out to see if that will prevent this overfitting.  

### Citations:

GeeksforGeeks. "Build a Recommendation Engine with Collaborative Filtering." GeeksforGeeks, GeeksforGeeks, https://www.geeksforgeeks.org/build-a-recommendation-engine-with-collaborative-filtering/

Chen, Yihong. Neural Collaborative Filtering. GitHub, https://github.com/yihong-chen/neural-collaborative-filtering.